# Mutation Analysis Notebook

This notebook corresponds to the analysis part of `Untitled.ipynb` Cell 0 + Cell 2 and `mutation-analysis.ipynb` Cells 1/3.

It selects correctly predicted sequences, generates single-point mutants, and predicts the mutated sequences with a fine-tuned LAMAR-m6A model. Plotting is handled separately.

## Step 0. Project root

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "lamar_m6a").exists():
    raise RuntimeError("Cannot find project root containing lamar_m6a/. Run this notebook inside the project directory.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/picb/rnasys2/lidaoqian/LAMAR-M6A')

## Step 1. Import analysis function

In [5]:
from lamar_m6a.interpretability.run_mutation_analysis import run_mutation_analysis
from argparse import Namespace

## Step 2. Configure inputs and outputs

The input prediction CSV should contain `seq`, `label`, and either `predict_label` or `predict`.

In [ ]:
MODE = "full"
TEST_CELL = "cell5"      # 被分析的测试集来自哪个 cell
MODEL_CELL = "cell5"      # 使用哪个 fine-tuned 模型
LENGTH = 801
LABEL = 1
MAX_SEQUENCES = 100

INPUT_CSV = PROJECT_ROOT / "outputs" / "predictions" / "single_csv" / MODE / f"model_{MODEL_CELL}_target_{TEST_CELL}_{LENGTH}_test_pred.csv"
MUTATION_CSV = PROJECT_ROOT / "outputs" / "interpretability" / "mutation" / f"{TEST_CELL}_mutated_sequences_{LENGTH}_label{LABEL}_{MAX_SEQUENCES}.csv"
OUTPUT_CSV = PROJECT_ROOT / "outputs" / "interpretability" / "mutation" / f"{TEST_CELL}_mutated_sequences_{LENGTH}_label{LABEL}_{MAX_SEQUENCES}_predict_linear.csv"

CHECKPOINT_PATH = PROJECT_ROOT / "outputs" / "finetune" / MODE / f"{MODEL_CELL}-{LENGTH}-bs16-acc4"
TOKENIZER_PATH = PROJECT_ROOT / "tokenizer" / "single_nucleotide"
MODEL_CONFIG = PROJECT_ROOT / "config" / "config_150M.json"
MODEL_CLASS = "scripts.sequence_classification_patch:EsmForSequenceClassification"

INPUT_CSV, MUTATION_CSV, OUTPUT_CSV, CHECKPOINT_PATH

## Step 3. Check paths

In [ ]:
for path in [INPUT_CSV, CHECKPOINT_PATH, TOKENIZER_PATH, MODEL_CONFIG]:
    print(path, path.exists())

## Step 4. Run mutation analysis

In [ ]:
args = Namespace(
    input_csv=INPUT_CSV,
    mutation_csv=MUTATION_CSV,
    output_csv=OUTPUT_CSV,
    checkpoint_path=CHECKPOINT_PATH,
    label=LABEL,
    predicted_label=1,
    max_sequences=MAX_SEQUENCES,
    seed=42,
    skip_center_a=False,
    allow_n=False,
    length=LENGTH,
    tokenizer_path=TOKENIZER_PATH,
    model_config=MODEL_CONFIG,
    model_class=MODEL_CLASS,
    model_max_length=LENGTH + 2,
    padding_side="left",
    use_fast_tokenizer=False,
    positional_embedding_type="rotary",
    hidden_size=768,
    intermediate_size=3072,
    num_attention_heads=12,
    num_hidden_layers=12,
    head_type="Linear",
    kernel_sizes=None,
    ocs=None,
    strict_load=True,
    batch_size=512,
    device=None,
)

run_mutation_analysis(args)

## Step 5. Output check

In [ ]:
import pandas as pd

for path in [MUTATION_CSV, OUTPUT_CSV]:
    print(path, path.exists())
    if path.exists():
        df = pd.read_csv(path)
        print(df.head())
        print(df.shape)